# BERT

# Task 1 — Training BERT from Scratch (MLM + NSP)

Course: AT82.05 Artificial Intelligence: Natural Language Understanding (NLU)  
Assignment: A4 — Do you AGREE? (Updated 19 Jan 2026)

This notebook implements **BERT pretraining from scratch** using:
- **Masked Language Modeling (MLM)**
- **Next Sentence Prediction (NSP)**

We train on a **subset** of a reputable public dataset and save the trained weights for Task 2.


In [1]:
import os
import re
import time
import math
import pickle
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from datasets import load_dataset
from random import randrange, shuffle, randint
import time

d:\Ait\SEM 2\Nlp\assignment 4\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
print("torch cuda available:", torch.cuda.is_available()) # Check if CUDA is available
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Use GPU if available, else fallback to CPU
print("device:", device)

# Reproducibility
seed = 42 
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


torch cuda available: True
gpu: NVIDIA GeForce RTX 3060 Laptop GPU
device: cuda


## 1. Data
## Dataset (Reputable Source)

We use **WikiText-103 (raw)** via the **Hugging Face Datasets** repository.

- Dataset: WikiText-103 (Merity et al., 2016)
- Hugging Face config used: `wikitext-103-raw-v1`
- Split: `train`
- Subset used: first **100,000** non-empty text lines (as allowed by the assignment requirement to use a subset)

**Source / credit:**
- Hugging Face dataset entry: https://huggingface.co/datasets/wikitext  
- Original WikiText paper: *Pointer Sentinel Mixture Models* (Merity et al., 2016)


In [ ]:
raw = load_dataset("wikitext", "wikitext-103-raw-v1", split="train") # load the raw version (not tokenized)
texts = [t for t in raw["text"] if t and len(t.strip()) > 0] # filter out empty lines

# use a subset ~100k lines (adjust if you want)
texts = texts[:100000]

print("num texts:", len(texts))
print("sample:", texts[0][:200])

num texts: 100000
sample:  = Valkyria Chronicles III = 



## 2. Preprocessing

### Tokenization and numericalization
## Preprocessing Summary

Steps:
1. Remove empty lines from the raw dataset.
2. Convert text lines into token sequences.
3. Build vocabulary with special tokens: `[PAD]`, `[CLS]`, `[SEP]`, `[MASK]`.
4. Create `token_list` used for MLM+NSP batch generation.


In [ ]:
def basic_tokenize(s: str): 
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.split()

tokenized = [basic_tokenize(t) for t in texts] 
tokenized = [s for s in tokenized if 5 <= len(s) <= 60]

print("usable sentences:", len(tokenized))
print("example tokens:", tokenized[0][:10])


usable sentences: 20963
example tokens: ['on', 'its', 'day', 'of', 'release', 'in', 'japan', 'valkyria', 'chronicles', 'iii']


In [ ]:
#making vocabs - numericalization
specials = ['[PAD]', '[CLS]', '[SEP]', '[MASK]', '[UNK]']  # special tokens for padding, classification, separation, masking, and unknown words
word2id = {w:i for i,w in enumerate(specials)} # initialize word2id with special tokens

for sent in tokenized: # loop through each sentence in the tokenized list
    for w in sent:
        if w not in word2id:
            word2id[w] = len(word2id)

id2word = {i:w for w,i in word2id.items()} # create a reverse mapping from indices to words using a dictionary comprehension
vocab_size = len(word2id) # calculate the size of the vocabulary by taking the length of the word2id dictionary, which now contains all unique words from the tokenized sentences plus the special tokens

pad_id  = word2id['[PAD]'] # get the index of the padding token from the word2id dictionary and store it in pad_id for later use in padding sequences to a fixed length
cls_id  = word2id['[CLS]'] # get the index of the classification token from the word2id dictionary and store it in cls_id for later use in marking the beginning of a sequence for classification tasks
sep_id  = word2id['[SEP]'] # get the index of the separation token from the word2id dictionary and store it in sep_id for later use in marking the separation between two sequences in tasks like sentence pair classification
mask_id = word2id['[MASK]'] # get the index of the masking token from the word2id dictionary and store it in mask_id for later use in masking tokens during training for masked language modeling tasks
unk_id  = word2id['[UNK]'] # get the index of the unknown token from the word2id dictionary and store it in unk_id for later use in representing words that are not in the vocabulary during tokenization and numericalization processes

IGNORE_INDEX = -100

print("vocab_size:", vocab_size)


vocab_size: 45891


In [ ]:
os.makedirs("models", exist_ok=True) # create a directory named "models" if it doesn't already exist to store the vocabulary file

with open("models/vocab.pkl", "wb") as f: # open a file named "vocab.pkl" in the "models" directory in write-binary mode and assign it to the variable f
    pickle.dump({"word2id": word2id, "id2word": id2word}, f)

print("saved vocab to models/vocab.pkl")

saved vocab to models/vocab.pkl


In [ ]:
# Convert tokens to ids
token_list = [ # create a new list called token_list that will contain the numericalized sentences
    [word2id.get(w, word2id['[UNK]']) for w in sent]
    for sent in tokenized
]

sentences = tokenized  # store the original tokenized sentences in a variable called sentences for later use in training and evaluation

print("token_list length:", len(token_list))
print("first tokenized sentence (ids):", token_list[0][:10])


token_list length: 20963
first tokenized sentence (ids): [5, 6, 7, 8, 9, 10, 11, 12, 13, 14]


## 3. Batch generation (random sampling for MLM + NSP)

## Model Configuration (BERT Pretraining)

Architecture (Transformer encoder):
- Layers (`n_layers`): 4
- Attention heads (`n_heads`): 4
- Hidden size (`d_model`): 256
- Feedforward size (`d_ff`): 1024

Training setup:
- Max sequence length (`max_len`): 128
- Batch size (`batch_size`): 32
- Max masked tokens (`max_mask`): 20
- Masking ratio: 15% of tokens (capped by `max_mask`)
- Optimizer: Adam
- Learning rate: 3e-4
- Loss: CrossEntropyLoss (ignore_index = IGNORE_INDEX for padded MLM targets)


In [ ]:
# Model configuration 
n_layers = 4 # number of transformer layers in the BERT model
n_heads  = 4 # number of attention heads in the multi-head self-attention mechanism of the transformer layers, which allows the model to attend to different parts of the input sequence simultaneously and capture various types of relationships between tokens for better contextual understanding and representation learning in tasks like masked language modeling and next sentence prediction.
d_model  = 256 # dimensionality of the input and output embeddings in the transformer layers, which determines the size of the hidden representations learned by the model and affects its capacity to capture complex patterns and relationships in the data for tasks like masked language modeling and next sentence prediction.
d_ff     = 1024 # dimensionality of the feed-forward neural network in the transformer layers, which is typically larger than d_model to allow for more expressive power and non-linearity in the model's representations, and helps the model learn complex transformations of the input data for tasks like masked language modeling and next sentence prediction.
d_k = d_v = d_model // n_heads # dimensionality of the key and value vectors in the multi-head self-attention mechanism, which is calculated by dividing the model dimension (d_model) by the number of attention heads (n_heads) to ensure that the total dimensionality of the concatenated attention outputs from all heads matches the model dimension for proper integration into the transformer layers during tasks like masked language modeling and next sentence prediction.

max_len = 128 # maximum sequence length for the input sentences, which determines the maximum number of tokens that can be processed by the model in a single forward pass, and affects the computational efficiency and memory requirements of the model during training and inference for tasks like masked language modeling and next sentence prediction.
batch_size = 32 # number of samples processed together in one forward and backward pass during training, which affects the stability of the training process, the convergence speed, and the generalization performance of the model for tasks like masked language modeling and next sentence prediction.
max_mask = 20 # maximum number of tokens to mask in each input sequence during training for masked language modeling, which determines the level of difficulty and the amount of information the model needs to learn to predict the masked tokens correctly, and can be adjusted based on the average length of the input sentences and the desired training dynamics for tasks like masked language modeling.
n_segments = 2 # number of segments to use in the input sequences for tasks like next sentence prediction, which determines how the input sentences are divided into segments and how the segment embeddings are used to differentiate between them, affecting the model's ability to learn relationships between sentences and capture contextual information for tasks like masked language modeling and next sentence prediction.
mlm_prob = 0.15 # probability of masking each token in the input sequence during training for masked language modeling, which determines the proportion of tokens that will be masked and need to be predicted by the model, affecting the difficulty of the training task and the amount of information the model learns to capture about the context and relationships between tokens for tasks like masked language modeling.


## Implementation Notes

To improve training efficiency **without changing BERT objectives**:

1. **Mask position selection:** use uniform sampling of masked positions (equivalent distribution to shuffling candidate positions).
2. **NSP pair generation:** precompute valid positive (true next) and negative (random) sentence index pairs to avoid slow rejection sampling during batch creation.

These changes keep the MLM+NSP training objective the same and only reduce CPU overhead in data preparation.


In [9]:
# --- Precompute NSP pairs for fast batch generation ---
N = len(token_list)
assert N >= 2, "token_list is too small to build NSP pairs."

pos_pairs = []  # (a_idx, b_idx) where b is the true next sentence
neg_pairs = []  # (a_idx, b_idx) where b is a random sentence (not next)

# Positive pairs: (i, i+1)
for i in range(N - 1):
    pos_pairs.append((i, i + 1))

# Negative pairs: (i, j) where j != i+1
for i in range(N - 1):
    j = random.randrange(N)
    while j == i + 1:  # avoid true-next
        j = random.randrange(N)
    neg_pairs.append((i, j))

print("Pairs prepared:",
      "pos =", len(pos_pairs),
      "neg =", len(neg_pairs))


Pairs prepared: pos = 20962 neg = 20962


In [ ]:
def make_batch(): # function to create a batch of training data for masked language modeling and next sentence prediction tasks, which involves sampling positive and negative pairs of sentences, applying masking to the input tokens, and preparing the input ids, segment ids, masked tokens, masked positions, and labels for training the BERT model.
    batch = []

    half = batch_size // 2

    # sample half positive, half negative (same requirement as before)
    pos_samples = random.sample(pos_pairs, half)
    neg_samples = random.sample(neg_pairs, half)

    # label 1 = IsNext, label 0 = NotNext
    samples = [(a, b, 1) for (a, b) in pos_samples] + [(a, b, 0) for (a, b) in neg_samples]
    random.shuffle(samples)  # mix them

    for a_idx, b_idx, label in samples:
        tokens_a = token_list[a_idx]
        tokens_b = token_list[b_idx]

        # 1) token ids with specials
        input_ids = [word2id['[CLS]']] + tokens_a + [word2id['[SEP]']] + tokens_b + [word2id['[SEP]']]

        # 2) segment ids
        segment_ids = [0] * (1 + len(tokens_a) + 1) + [1] * (len(tokens_b) + 1)

        # 2.5) truncate to max_len
        input_ids = input_ids[:max_len]
        segment_ids = segment_ids[:max_len]

        # 3) masking: 15% with cap max_mask
        n_pred = min(max_mask, max(1, int(round(len(input_ids) * 0.15))))
        cand_pos = [i for i, token in enumerate(input_ids)
                    if token != word2id['[CLS]'] and token != word2id['[SEP]']]

        masked_tokens, masked_pos = [], []
        for pos in random.sample(cand_pos, k=min(n_pred, len(cand_pos))):
            masked_pos.append(pos)
            masked_tokens.append(input_ids[pos])

            p = random.random()
            if p < 0.8:        # 80% mask
                input_ids[pos] = word2id['[MASK]']
            elif p < 0.9:      # 10% random token
                input_ids[pos] = randint(0, vocab_size - 1)
            else:              # 10% unchanged
                pass

        # 4) pad to max_len
        n_pad = max_len - len(input_ids)
        input_ids.extend([word2id['[PAD]']] * n_pad)
        segment_ids.extend([0] * n_pad)

        # 5) pad masked lists to max_mask
        if len(masked_tokens) < max_mask:
            pad = max_mask - len(masked_tokens)
            masked_tokens.extend([IGNORE_INDEX] * pad)
            masked_pos.extend([0] * pad)

        batch.append([input_ids, segment_ids, masked_tokens, masked_pos, label])

    return batch


In [ ]:
batch = make_batch() # call the make_batch function to generate a batch of training data for masked language modeling and next sentence prediction tasks, which will be used for training the BERT model.
input_ids, segment_ids, masked_tokens, masked_pos, isNext = map(torch.LongTensor, zip(*batch))
print(input_ids.shape, segment_ids.shape, masked_tokens.shape, masked_pos.shape, isNext.shape)


torch.Size([32, 128]) torch.Size([32, 128]) torch.Size([32, 20]) torch.Size([32, 20]) torch.Size([32])


### Optimization note

- Mask positions are selected using `random.sample` rather than shuffling the entire list of candidates.
- NSP batches are generated by sampling from precomputed positive/negative index pairs.

These changes preserve MLM+NSP training objectives and sampling proportions while reducing CPU overhead.


In [ ]:
t0 = time.time() # measure the time taken to generate batches of training data by calling the make_batch function multiple times and calculating the average time per batch in milliseconds, which can help evaluate the efficiency of the batch generation process and identify potential bottlenecks for optimization during training of the BERT model.
for _ in range(10):
    _ = make_batch()
print("make_batch avg ms:", (time.time()-t0)/10 * 1000)


make_batch avg ms: 1.5996694564819336


## 4. Model

Recall that BERT only uses the encoder.

BERT has the following components:

- Embedding layers
- Attention Mask
- Encoder layer
- Multi-head attention
- Scaled dot product attention
- Position-wise feed-forward network
- BERT (assembling all the components)

## 4.1 Embedding

<img src = "../figures/BERT_embed.png" width=500>

In [13]:
class Embedding(nn.Module):
    def __init__(self):
        super(Embedding, self).__init__()
        self.tok_embed = nn.Embedding(vocab_size, d_model)  # token embedding
        self.pos_embed = nn.Embedding(max_len, d_model)      # position embedding
        self.seg_embed = nn.Embedding(n_segments, d_model)  # segment(token type) embedding
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, seg):
        #x, seg: (bs, len)
        seq_len = x.size(1)
        pos = torch.arange(seq_len, dtype=torch.long, device=x.device)
        pos = pos.unsqueeze(0).expand_as(x)  # (len,) -> (bs, len)
        embedding = self.tok_embed(x) + self.pos_embed(pos) + self.seg_embed(seg)
        return self.norm(embedding)

## 4.2 Attention mask

In [14]:
def get_attn_pad_mask(seq_q, seq_k):
    batch_size, len_q = seq_q.size()
    batch_size, len_k = seq_k.size()
    # PAD token masking
    pad_attn_mask = seq_k.data.eq(pad_id).unsqueeze(1)  # (bs, 1, len_k)
    return pad_attn_mask.expand(batch_size, len_q, len_k)  # (bs, len_q, len_k)


### Testing the attention mask

In [15]:
print(get_attn_pad_mask(input_ids, input_ids).shape)

torch.Size([32, 128, 128])


## 4.3 Encoder

The encoder has two main components: 

- Multi-head Attention
- Position-wise feed-forward network


Let's define the scaled dot attention, to be used inside the multihead attention

In [16]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()

    def forward(self, Q, K, V, attn_mask):
        scores = torch.matmul(Q, K.transpose(-1, -2)) / np.sqrt(d_k) # scores : [batch_size x n_heads x len_q(=len_k) x len_k(=len_q)]
        scores.masked_fill_(attn_mask, -1e9) # Fills elements of self tensor with value where mask is one.
        attn = nn.Softmax(dim=-1)(scores)
        context = torch.matmul(attn, V)
        return context, attn 

Here is the Multiheadattention.

In [17]:
class MultiHeadAttention(nn.Module):
    def __init__(self):
        super(MultiHeadAttention, self).__init__()
        self.W_Q = nn.Linear(d_model, d_k * n_heads)
        self.W_K = nn.Linear(d_model, d_k * n_heads)
        self.W_V = nn.Linear(d_model, d_v * n_heads)

        self.fc = nn.Linear(n_heads * d_v, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, Q, K, V, attn_mask):
        residual, batch_size = Q, Q.size(0)

        q_s = self.W_Q(Q).view(batch_size, -1, n_heads, d_k).transpose(1, 2)
        k_s = self.W_K(K).view(batch_size, -1, n_heads, d_k).transpose(1, 2)
        v_s = self.W_V(V).view(batch_size, -1, n_heads, d_v).transpose(1, 2)

        attn_mask = attn_mask.unsqueeze(1).repeat(1, n_heads, 1, 1)

        context, attn = ScaledDotProductAttention()(q_s, k_s, v_s, attn_mask)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, n_heads * d_v)

        output = self.fc(context)
        return self.norm(output + residual), attn


Here is the PoswiseFeedForwardNet.

In [18]:
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self):
        super(PoswiseFeedForwardNet, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # (batch_size, len_seq, d_model) -> (batch_size, len_seq, d_ff) -> (batch_size, len_seq, d_model)
        return self.fc2(F.gelu(self.fc1(x)))


In [19]:
class EncoderLayer(nn.Module):
    def __init__(self):
        super(EncoderLayer, self).__init__()
        self.enc_self_attn = MultiHeadAttention()
        self.pos_ffn       = PoswiseFeedForwardNet()

    def forward(self, enc_inputs, enc_self_attn_mask):
        enc_outputs, attn = self.enc_self_attn(enc_inputs, enc_inputs, enc_inputs, enc_self_attn_mask) # enc_inputs to same Q,K,V
        enc_outputs = self.pos_ffn(enc_outputs) # enc_outputs: [batch_size x len_q x d_model]
        return enc_outputs, attn

## 4.4 Putting them together

In [20]:
class BERT(nn.Module):
    def __init__(self):
        super(BERT, self).__init__()
        self.embedding = Embedding()
        self.layers = nn.ModuleList([EncoderLayer() for _ in range(n_layers)])
        self.fc = nn.Linear(d_model, d_model)
        self.activ = nn.Tanh()
        self.linear = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, 2)
        # decoder is shared with embedding layer
        embed_weight = self.embedding.tok_embed.weight
        n_vocab, n_dim = embed_weight.size()
        self.decoder = nn.Linear(n_dim, n_vocab, bias=False)
        self.decoder.weight = embed_weight
        self.decoder_bias = nn.Parameter(torch.zeros(n_vocab))

    def forward(self, input_ids, segment_ids, masked_pos=None, return_sequence_output=False):
        output = self.embedding(input_ids, segment_ids)
        enc_self_attn_mask = get_attn_pad_mask(input_ids, input_ids)
        for layer in self.layers:
            output, enc_self_attn = layer(output, enc_self_attn_mask)
        # output : [batch_size, len, d_model], attn : [batch_size, n_heads, d_mode, d_model]
        if return_sequence_output:
            attention_mask = (input_ids != pad_id).long()
            return output, attention_mask
        # 1. predict next sentence
        # it will be decided by first token(CLS)
        h_pooled   = self.activ(self.fc(output[:, 0])) # [batch_size, d_model]
        logits_nsp = self.classifier(h_pooled) # [batch_size, 2]

        # 2. predict the masked token
        if masked_pos is None:
            raise ValueError("masked_pos must be provided when return_sequence_output=False")
        masked_pos = masked_pos[:, :, None].expand(-1, -1, output.size(-1)) # [batch_size, max_pred, d_model]
        h_masked = torch.gather(output, 1, masked_pos) # masking position [batch_size, max_pred, d_model]
        h_masked  = self.norm(F.gelu(self.linear(h_masked)))
        logits_lm = self.decoder(h_masked) + self.decoder_bias # [batch_size, max_pred, n_vocab]

        return logits_lm, logits_nsp

Forward Sanity test

In [ ]:
model = BERT().to(device) # instantiate the BERT model and move it to the specified device (GPU if available, otherwise CPU) for training and inference.

# move batch tensors to GPU
input_ids_gpu = input_ids.to(device)
segment_ids_gpu = segment_ids.to(device)
masked_pos_gpu = masked_pos.to(device)

logits_lm, logits_nsp = model(input_ids_gpu, segment_ids_gpu, masked_pos_gpu)

print("logits_lm shape:", logits_lm.shape)
print("logits_nsp shape:", logits_nsp.shape)


logits_lm shape: torch.Size([32, 20, 45891])
logits_nsp shape: torch.Size([32, 2])


## 5. Training

# We are NOT using a PyTorch DataLoader that iterates through the dataset sequentially.
# Instead, each training step calls make_batch(), which randomly samples sentence pairs
# from the corpus and dynamically applies MLM masking + NSP labeling.
# Therefore, one "epoch" in this notebook is defined as a fixed number of update steps
# (steps_per_epoch), not a full pass over the entire dataset.
# Total parameter updates = num_epochs * steps_per_epoch
## Training Details

We train on GPU (CUDA when available).

- Epochs: 5
- Steps per epoch: 2000
- Total update steps: 10,000
- Print frequency: every 200 steps



In [22]:
## 5. Training (MLM + NSP)

num_epochs = 5 # number of epochs to train
steps_per_epoch = 2000 # number of randomly-sampled batches per epoch

model = BERT().to(device) # move model to GPU if available
print("Training on:", next(model.parameters()).device) # check device of model parameters

criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX) # loss function for both MLM and NSP
optimizer = optim.Adam(model.parameters(), lr=3e-4) # optimizer for training

model.train() # set model to training mode

for epoch in range(1, num_epochs + 1): # loop over epochs
    total_loss = 0.0
    total_lm = 0.0
    total_nsp = 0.0

    t0 = time.time()           # timer for print interval
    epoch_start = time.time()  # timer for whole epoch

    for step in range(1, steps_per_epoch + 1): # loop over steps in the epoch
        batch = make_batch()
        input_ids, segment_ids, masked_tokens, masked_pos, isNext = map(torch.LongTensor, zip(*batch))

        input_ids = input_ids.to(device)
        segment_ids = segment_ids.to(device)
        masked_tokens = masked_tokens.to(device)
        masked_pos = masked_pos.to(device)
        isNext = isNext.to(device)

        optimizer.zero_grad()

        logits_lm, logits_nsp = model(input_ids, segment_ids, masked_pos)

        loss_lm = criterion(logits_lm.transpose(1, 2), masked_tokens).mean()
        loss_nsp = criterion(logits_nsp, isNext)
        loss = loss_lm + loss_nsp

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_lm += loss_lm.item()
        total_nsp += loss_nsp.item()

        if step % 200 == 0: # print every 200 steps
            dt = time.time() - t0
            print(
                f"Epoch {epoch}/{num_epochs} Step {step}/{steps_per_epoch} | "
                f"loss={loss.item():.4f} | sec/step={dt/200:.3f}",
                flush=True
            )
            t0 = time.time() 

    epoch_time = time.time() - epoch_start # total time for the epoch
    print(
        f"Epoch {epoch} DONE | avg_loss={total_loss/steps_per_epoch:.4f} | "
        f"avg_lm={total_lm/steps_per_epoch:.4f} | avg_nsp={total_nsp/steps_per_epoch:.4f} | "
        f"epoch_time={epoch_time/60:.2f} min | avg_sec/step={epoch_time/steps_per_epoch:.3f}",
        flush=True
    )


Training on: cuda:0
Epoch 1/5 Step 200/2000 | loss=24.6501 | sec/step=0.053
Epoch 1/5 Step 400/2000 | loss=16.4771 | sec/step=0.047
Epoch 1/5 Step 600/2000 | loss=14.0605 | sec/step=0.047
Epoch 1/5 Step 800/2000 | loss=13.0744 | sec/step=0.046
Epoch 1/5 Step 1000/2000 | loss=11.0387 | sec/step=0.047
Epoch 1/5 Step 1200/2000 | loss=10.5068 | sec/step=0.046
Epoch 1/5 Step 1400/2000 | loss=9.9000 | sec/step=0.046
Epoch 1/5 Step 1600/2000 | loss=9.2590 | sec/step=0.046
Epoch 1/5 Step 1800/2000 | loss=9.8081 | sec/step=0.046
Epoch 1/5 Step 2000/2000 | loss=8.9507 | sec/step=0.046
Epoch 1 DONE | avg_loss=14.4466 | avg_lm=13.7603 | avg_nsp=0.6863 | epoch_time=1.57 min | avg_sec/step=0.047
Epoch 2/5 Step 200/2000 | loss=8.9614 | sec/step=0.046
Epoch 2/5 Step 400/2000 | loss=8.4691 | sec/step=0.046
Epoch 2/5 Step 600/2000 | loss=8.6300 | sec/step=0.046
Epoch 2/5 Step 800/2000 | loss=9.2583 | sec/step=0.046
Epoch 2/5 Step 1000/2000 | loss=8.1497 | sec/step=0.046
Epoch 2/5 Step 1200/2000 | loss=8

## Training Results

Training loss decreased over epochs, indicating the model is learning MLM.

Example from this run:
- Epoch 1 avg_loss ≈ 14.45
- Epoch 5 avg_loss ≈ 7.91

NSP loss decreased slightly which may reflect that WikiText lines are not always clean sentence boundaries.


In [ ]:
ckpt = { # create a checkpoint dictionary to save the model state and configuration for later use in inference or further training, which includes the model's state dictionary containing the learned parameters, the configuration of the model architecture, and the vocabulary mappings for tokenization and numericalization.
    "model_state_dict": model.state_dict(),
    "config": {
        "n_layers": n_layers,
        "n_heads": n_heads,
        "d_model": d_model,
        "d_ff": d_ff,
        "d_k": d_k,
        "d_v": d_v,
        "max_len": max_len,
        "max_mask": max_mask,
        "n_segments": n_segments,
        "vocab_size": vocab_size,
        "pad_id": pad_id,
        "cls_id": cls_id,
        "sep_id": sep_id,
        "mask_id": mask_id,
        "unk_id": unk_id,
    },
    "vocab": {"word2id": word2id, "id2word": id2word},
}

torch.save(ckpt, "models/bert_task1_ckpt.pt")
print("Saved bundle to models/bert_task1_ckpt.pt")


Saved bundle to models/bert_task1_ckpt.pt


## Saving Trained Weights (for Task 2)

We save:
1. Model weights (`state_dict`) for reuse in Sentence-BERT training (Task 2)
2. (Optional) Full checkpoint bundle with optimizer state for resuming training


In [ ]:
path = "models/bert_task1_ckpt.pt" # check if the checkpoint file exists and print its existence and size in bytes, which can help verify that the model checkpoint was saved successfully and provide information about the storage requirements for the checkpoint file.
if os.path.exists(path):
    print("ckpt exists:True | size", os.path.getsize(path))
else:
    print("ckpt exists:False")


ckpt exists:True | size 61789140


In [25]:
os.makedirs("weights", exist_ok=True)
torch.save(model.state_dict(), "weights/bert_task1.pt")
print("Saved: weights/bert_task1.pt")



Saved: weights/bert_task1.pt


In [26]:
#sanity check - get sequence output and attention mask
model.eval()
batch = make_batch()
input_ids, segment_ids, masked_tokens, masked_pos, isNext = map(torch.LongTensor, zip(*batch))
input_ids = input_ids.to(device)
segment_ids = segment_ids.to(device)

with torch.no_grad():
    seq_out, attn_mask = model(input_ids, segment_ids, return_sequence_output=True)

print("seq_out:", seq_out.shape)      # (bs, max_len, d_model)
print("attn_mask:", attn_mask.shape) # (bs, max_len)
print("nonpad avg:", attn_mask.sum(dim=1).float().mean().item())


seq_out: torch.Size([32, 128, 256])
attn_mask: torch.Size([32, 128])
nonpad avg: 56.84375


In [27]:
print("Training hyperparameters:")
print("num_epochs =", num_epochs)
print("steps_per_epoch =", steps_per_epoch)
print("batch_size =", batch_size)
print("max_len =", max_len)
print("vocab_size =", vocab_size)
print("lr =", 3e-4)


Training hyperparameters:
num_epochs = 5
steps_per_epoch = 2000
batch_size = 32
max_len = 128
vocab_size = 45891
lr = 0.0003


## 6. Inference
This section performs a small qualitative demonstration of MLM and NSP predictions.
The model is trained on a limited subset and limited number of update steps, so results are not comparable to fully pretrained BERT models.

In [28]:
# ===== Inference / quick demo (MLM + NSP) =====
model.eval()

# Make a fresh batch (so we have a valid sample)
batch = make_batch()
input_ids, segment_ids, masked_tokens, masked_pos, isNext = map(torch.LongTensor, zip(*batch))

# pick one sample from the batch
i = 0
inp = input_ids[i:i+1].to(device)
seg = segment_ids[i:i+1].to(device)
mpos = masked_pos[i:i+1].to(device)
mtok = masked_tokens[i:i+1]  # keep on CPU for printing
label = isNext[i].item()

with torch.no_grad():
    logits_lm, logits_nsp = model(inp, seg, mpos)

# NSP prediction
pred_nsp = logits_nsp.argmax(dim=1).item()
print("NSP true:", label, "| NSP pred:", pred_nsp, "(1=IsNext, 0=NotNext)")

# Show the original (masked) input tokens
inp_cpu = inp[0].detach().cpu().tolist()
tokens_inp = [id2word[t] for t in inp_cpu if t != word2id['[PAD]']]
print("\nInput tokens (with [MASK]):")
print(tokens_inp)

# MLM prediction for masked positions
pred_ids = logits_lm.argmax(dim=2)[0].detach().cpu().tolist()  # (max_mask,)
true_ids = mtok[0].tolist()
pos_ids  = masked_pos[i].tolist()

print("\nMasked positions / true / predicted:")
for j in range(max_mask):
    pos = pos_ids[j]
    if true_ids[j] == IGNORE_INDEX or pos == 0:
        continue
    true_w = id2word[true_ids[j]]
    pred_w = id2word[pred_ids[j]]
    print(f"pos={pos:3d} | true={true_w:15s} | pred={pred_w:15s}")


NSP true: 0 | NSP pred: 1 (1=IsNext, 0=NotNext)

Input tokens (with [MASK]):
['[CLS]', '[MASK]', '[MASK]', 'written', 'by', 'solo', 'vocals', 'peignot', '[SEP]', 'decorative', 'asomtavruli', 'capital', 'letters', 'm', 'n', 'and', 't', '12', '13th', 'century', '[SEP]']

Masked positions / true / predicted:
pos=  1 | true=michael         | pred=the            
pos=  7 | true=percussion      | pred=the            
pos=  2 | true=jackson         | pred=the            


## Sanity check (overfit one batch)

We intentionally overfit on a single fixed batch to verify:
- the forward pass is correct,
- MLM targets align with logits,
- gradients flow through attention and embeddings,
- training loop + optimizer work correctly.

A rapid loss drop on this fixed batch is expected and indicates the implementation is correct (this is not generalization performance).


In [29]:
# ===== Overfit sanity check on ONE fixed batch =====
model_overfit = BERT().to(device)
optimizer_overfit = optim.Adam(model_overfit.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)


fixed_batch = make_batch()
input_ids, segment_ids, masked_tokens, masked_pos, isNext = map(torch.LongTensor, zip(*fixed_batch))

input_ids = input_ids.to(device)
segment_ids = segment_ids.to(device)
masked_tokens = masked_tokens.to(device)
masked_pos = masked_pos.to(device)
isNext = isNext.to(device)

model_overfit.train()
for step in range(1, 301):
    optimizer_overfit.zero_grad()

    logits_lm, logits_nsp = model_overfit(input_ids, segment_ids, masked_pos)

    loss_lm = criterion(logits_lm.transpose(1, 2), masked_tokens).mean()
    loss_nsp = criterion(logits_nsp, isNext)
    loss = loss_lm + loss_nsp

    loss.backward()
    optimizer_overfit.step()

    if step % 50 == 0:
        with torch.no_grad():
            nsp_acc = (logits_nsp.argmax(dim=1) == isNext).float().mean().item()
        print(f"step {step} | loss {loss.item():.4f} | lm {loss_lm.item():.4f} | nsp {loss_nsp.item():.4f} | nsp_acc {nsp_acc:.2f}")


step 50 | loss 6.0497 | lm 5.3701 | nsp 0.6796 | nsp_acc 0.62
step 100 | loss 0.3012 | lm 0.0050 | nsp 0.2961 | nsp_acc 0.97
step 150 | loss 0.0016 | lm 0.0011 | nsp 0.0005 | nsp_acc 1.00
step 200 | loss 0.0008 | lm 0.0005 | nsp 0.0003 | nsp_acc 1.00
step 250 | loss 0.0006 | lm 0.0003 | nsp 0.0002 | nsp_acc 1.00
step 300 | loss 0.0004 | lm 0.0003 | nsp 0.0002 | nsp_acc 1.00


### Interpretation of Overfit Sanity Check

The rapid loss decrease on a fixed batch confirms:

- The forward pass is implemented correctly.
- MLM targets align with the predicted logits.
- Attention masking does not block gradients.
- Backpropagation and optimizer updates function properly.


## Task 1 Checklist

- [x] Implemented BERT encoder with MLM + NSP objectives from scratch  
- [x] Trained on reputable public dataset (WikiText-103 via Hugging Face Datasets) with proper credit  
- [x] Used subset of data (100k lines) as instructed  
- [x] Trained using CUDA when available  
- [x] Saved trained weights for Task 2 (`weights/bert_task1.pt`)  
